In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

<font size="2">
step 1: sanitisation :
load network traffic data, extract feature columns using positional slicing (`iloc`)

[ iloc is used to track the columns by their numerical column index rather than their names- to prevent typing or name tracking error- skipping text based search]

, and sanitize infinite/NaN values to prevent mathematical crashes.
</font>

In [38]:
# Step 1: Ingestion & Sanitation (Method 3 Position Slicing)

# Load dataset
df = pd.read_csv("cleaned_ids2018_sampled.csv")

# Extract 8 behavioral network features by column position
X_raw = df.iloc[:, 0:8].values ### the default 8 pipelines normally used 

# Extract target label (typically the last column)
y = df.iloc[:, -1].values

# Sanitize: replace missing (NaN) and infinite (Inf/-Inf) values to prevent division errors
X_clean = np.nan_to_num(X_raw, nan=0.0, posinf=0.0, neginf=0.0)## converting each anomolies into numerical value 0.0 to prevent errors


<font size="2">
step 2: Feature Standardization:
 z score normalization- <u>average=0 and standard deviation=1</u>
 <br>

 so high-magnitude metrics don't dominate PCA(Principal Component Analysis.) and to keep the math unit-independent.

 this helps us to check for outlies easily
</font>

In [39]:
# Step 2: Feature Standardization

# Scale features so no high-magnitude metric dominates PCA
scaler = StandardScaler() 
X_scaled = scaler.fit_transform(X_clean)


<font size="2">

step 3: Dimensionality Reduction (PCA)
<br>
Compress the 8 network features into 3 principal components to optimize computational efficiency while retaining maximum dataset variance.

</font>

In [40]:
# Step 3: Dimensionality Reduction (PCA)

# Compress 8 behavioral dimensions to 3 components for computational efficiency
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)

retained_var = np.sum(pca.explained_variance_ratio_) * 100
print(f" PCA Matrix Shape: {X_pca.shape}")
print(f" Retained Variance: {retained_var:.2f}%\n")


 PCA Matrix Shape: (1252846, 3)
 Retained Variance: 52.05%



<font size="2">
step 4:Vectorized GMM Modeling & Anomaly Scoring

Train Model: Fit a Gaussian Mixture Model (GMM) on the 3 PCA features to learn normal network behavior.

Calculate Scores: Measure how well each network flow fits this normal pattern (log-likelihood score).

Flag Anomalies: Mark the lowest 5% of scores as potential cyberattacks.
</font>

In [41]:
# Step 4: Vectorized GMM Modeling & Anomaly Scoring

# Fit a unified GMM model directly on the reduced feature space
gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
gmm.fit(X_pca)

# Vectorized computation of log-likelihood density scores (no row-by-row loops!)
log_likelihoods = gmm.score_samples(X_pca)#Normalcy Score- how likely is the network flow normal

# Set statistical threshold (lowest 5th percentile tail classified as anomalous)
threshold = np.percentile(log_likelihoods, 5)
predictions = np.where(log_likelihoods < threshold, "Anomaly", "Benign")


<font size ="2">
step 5: Results & Diagnostics Output
Calculate total anomaly counts and target class ratios across the dataset to evaluate detection yield.
</font>

In [42]:

unique_labels, label_counts = np.unique(predictions, return_counts=True)
total_samples = len(predictions)

print(" Anomaly Detection Results:")
for label, count in zip(unique_labels, label_counts):
    ratio = count / total_samples
    print(f"Label: {label:<8} | Count: {count:<6} | Ratio: {ratio:.4f} ({ratio*100:.2f}%)")

 Anomaly Detection Results:
Label: Anomaly  | Count: 62643  | Ratio: 0.0500 (5.00%)
Label: Benign   | Count: 1190203 | Ratio: 0.9500 (95.00%)


In [ ]:
import numpy as np

# 1. Get the scores and threshold
scores = gmm.score_samples(X_pca)
threshold2 = np.percentile(scores, 5)
sample_indices = np.random.choice(len(X_pca), 10000, replace=False)
X_sample = X_pca[sample_indices]
scores_sample = scores[sample_indices]
# 2. Separate anomalies from normal data
normal_points = X_pca[scores >= threshold2]
anomaly_points = X_pca[scores < threshold2]

# --- PLOT 1: Scatter Plot ---
plt.figure(figsize=(8, 5))
# Plot normal data in blue
plt.scatter(normal_points[:, 0], normal_points[:, 1], c='blue', alpha=0.3,s=20, label='Normal Traffic')
# Plot anomalies in pink
plt.scatter(anomaly_points[:, 0], anomaly_points[:, 1], c='pink', alpha=0.5,s=10, label='Anomalies')

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('GMM Anomalies in PCA Space')
plt.legend()
plt.show()


In [ ]:
# Preliminary GMM experiments with different K values
K_values = [2, 3, 4]
gmm_results = []

for K in K_values:
    gmm_temp = GaussianMixture(
        n_components=K,
        covariance_type='full',
        random_state=42,
        max_iter=200
    )
    gmm_temp.fit(X_pca)
    
    bic = gmm_temp.bic(X_pca)
    aic = gmm_temp.aic(X_pca)
    
    gmm_results.append({
        "K": K,
        "BIC": bic,
        "AIC": aic,
        "converged": gmm_temp.converged_
    })
## we are calculating BIC for each value of K to see which one is the best fit and which provide the most stable convergence
gmm_df = pd.DataFrame(gmm_results)
print("\nGMM model selection (preliminary):")
print(gmm_df)


GMM model selection (preliminary):
   K           BIC           AIC  converged
0  2  4.905188e+06  4.904960e+06       True
1  3 -5.236015e+06 -5.236364e+06       True
2  4 -5.385118e+06 -5.385587e+06       True
